# 01 · Business Understanding, Data Audit & Cleaning
*Sections 1–5 of the project: business understanding → dataset documentation → audit → cleaning decisions → feature engineering.*

## 1 · Business understanding
**The company.** A UK-based online giftware retailer. Its customers are a mix of **individual shoppers** and **wholesale / trade buyers** (many invoices contain 100+ units).
**The data.** Every transaction line between **1 Dec 2010 and 9 Dec 2011**.

**Decisions this analysis should support**
1. Where does the money come from — which products, customers, countries and moments?
2. How much revenue leaks through returns, and where?
3. Who should get loyalty perks, who needs a win-back offer, and who just needs a second purchase?

> *Guiding rule:* a cleaning step is only acceptable if we can say what it does to revenue.

In [1]:
import sys, warnings; sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, plotly.express as px, plotly.graph_objects as go
from src import cleaning as cl, features as ft, segmentation as sg, viz
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 40

## 2 · Dataset loading & documentation
| Column | Meaning | Gotcha |
|---|---|---|
| `InvoiceNo` | 6-digit invoice id | Prefix **C** = cancellation (return), prefix **A** = accounting adjustment. One invoice = many lines. |
| `StockCode` | product id | Contains non-products: `POST`, `DOT`, `M`, `BANK CHARGES`, `AMAZONFEE`… |
| `Description` | product name | 1,454 missing, mostly the zero-price rows |
| `Quantity` | units on the line | negative = return / write-off |
| `InvoiceDate` | timestamp | |
| `UnitPrice` | £ per unit | 0 = giveaway/error, negative = bad-debt adjustment |
| `CustomerID` | customer id | **25% missing** → guest checkouts. It is an *identifier*, never a numeric feature |
| `Country` | customer country | |

In [2]:
raw = cl.load_raw("../data/online_retail_raw.parquet")
print(raw.shape)
raw.head()

(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


## 3 · Data audit

In [3]:
cl.audit(raw)

,count
Rows,541909
Exact duplicate rows,5268
Missing CustomerID,135080
Missing Description,1454
Cancellation lines (InvoiceNo starts with C),9288
Adjustment invoices (InvoiceNo starts with A),3
Negative quantity lines,10624
Negative quantity but NOT a cancellation,1336
UnitPrice == 0,2515
UnitPrice < 0,2


**Reading the audit**
* 5,268 exact duplicate lines → remove (they would double count revenue).
* 9,288 cancellation lines, but **1,336 negative-quantity lines are *not* cancellations** (damages, stock write-offs, almost all price = 0 and guest) → they must not be treated as customer returns.
* 135,080 lines (25%) have no `CustomerID`. The guideline says *don't blindly drop* — so first we check how the gaps are distributed:

In [4]:
g = cl.guest_checkout_check(raw); g

{'invoices_total': 25900,
 'invoices_fully_guest': 3710,
 'invoices_fully_identified': 22190,
 'invoices_mixed': 0}

**Finding:** every invoice is either *fully* identified or *fully* anonymous (0 mixed invoices). So missing IDs are genuine **guest-checkout blocks** (3,710 invoices), not random data loss.
Decision → keep them for sales/product/time analytics (they are real revenue), exclude them only from customer-level RFM, where an identity is required.

## 4 · Cleaning & data-quality decisions (with revenue impact)

In [5]:
df, log = cl.clean(raw)
log.style.format({"Rows affected": "{:,}", "Revenue impact (GBP)": "£{:,.0f}"}).hide(axis="index")

Decision,Rows affected,Revenue impact (GBP),Business reason
Remove exact duplicate lines,"5,268","£21,741","Same invoice, product, qty, price, time -> scanner/export duplication, would double-count revenue."
"Flag: bad-debt adjustments (A*, negative price)",3,"£-11,062","Accounting entries, not customer purchases -> excluded from behaviour analysis."
Flag: zero-price lines,"2,510",£0,"Giveaways / write-offs / system errors -> no revenue, must not count as a purchase."
Flag: negative qty without 'C' prefix,"1,336",£0,"Damages, lost stock, manual corrections (mostly guests, price 0)."
"Flag: non-product codes (postage, fees, manual)","2,910","£-45,997","Real money, but not product demand -> kept in revenue, removed from product rankings."
Flag: cancellations (C*),"9,251","£-893,980",Returns. KEPT and netted against revenue instead of being dropped.
Flag: guest checkout (no CustomerID),"135,037","£1,447,488","Entire invoices are anonymous -> real sales, but cannot enter customer-level RFM."
"Genuine product sale lines (Qty>0, price>0, product code)","522,568","£10,247,905",Base for product & basket analytics.


**Why flags instead of deletes?** Every questionable line stays in the table with a boolean flag (`IsCancellation`, `IsZeroPrice`, `IsNonProduct`, `IsGuest`…). Each analysis then states exactly which flag combination it uses — nothing silently disappears, and the analysis stays reproducible.

### Returns: matched to originals, then netted
Guideline: *match cancellations with original invoices where possible.* Rule: same customer + same product, an earlier purchase exists, and the quantity bought ≥ quantity returned.

In [6]:
m = cl.match_returns(df)
print(f"Cancellation lines with a customer ID: {len(m):,}")
print(f"Matched to an earlier purchase of the same product by the same customer: {m['Matched'].mean():.1%}")
m.loc[~m["Matched"], ["InvoiceNo","StockCode","Description","Quantity","CustomerID"]].head(8)

Cancellation lines with a customer ID: 8,872
Matched to an earlier purchase of the same product by the same customer: 84.9%


,InvoiceNo,StockCode,Description,Quantity,CustomerID
0,C536379,D,Discount,-1,"14,527.00"
1,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,"15,311.00"
2,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,"17,548.00"
3,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,"17,548.00"
4,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,"17,548.00"
5,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,"17,548.00"
6,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,"17,548.00"
7,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,"17,548.00"


**85% of returns trace back to a real earlier purchase.** The remainder are mostly returns of items bought *before* 1 Dec 2010 (outside the window) — a data-window artefact, not an error. Returns are **kept and netted** (negative revenue), never deleted.

> ⚠️ **Phantom bestsellers.** Customer 16446 bought and cancelled 80,995 units of *Paper Craft, Little Birdie* (£168k) and another buyer did the same with 74,215 *Medium Ceramic Top Storage Jars*. On gross sales they are the #1 products by units. **Net of returns they vanish.** All product rankings in this project are therefore net.

In [7]:
pairs = df[df["StockCode"].isin(["23166","23843"])].sort_values("InvoiceDate")
pairs[["InvoiceNo","InvoiceDate","Description","Quantity","UnitPrice","CustomerID"]]

,InvoiceNo,InvoiceDate,Description,Quantity,UnitPrice,CustomerID
61006,541431,2011-01-18 10:01:00,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,"12,346.00"
61011,C541433,2011-01-18 10:17:00,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,"12,346.00"
185172,552882,2011-05-12 10:10:00,MEDIUM CERAMIC TOP STORAGE JAR,96,1.04,"14,646.00"
185597,552953,2011-05-12 12:11:00,MEDIUM CERAMIC TOP STORAGE JAR,4,1.25,"16,745.00"
186117,553005,2011-05-12 16:29:00,MEDIUM CERAMIC TOP STORAGE JAR,5,1.25,"14,651.00"
...,...,...,...,...,...,...
534519,581439,2011-12-08 16:30:00,MEDIUM CERAMIC TOP STORAGE JAR,2,2.46,NaN
535040,581476,2011-12-09 08:48:00,MEDIUM CERAMIC TOP STORAGE JAR,48,1.04,"12,433.00"
535160,581483,2011-12-09 09:15:00,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,"16,446.00"
535161,C581484,2011-12-09 09:27:00,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,"16,446.00"


## 5 · Feature engineering

In [8]:
# time features were created inside cl.clean(): Year, Month, YearMonth, DayOfWeek, Hour
# plus: TotalLineRevenue = Quantity x UnitPrice  and  Is_UK  (international vs home-market flag)
df[["InvoiceNo","InvoiceDate","Month","DayOfWeek","Hour","Quantity","UnitPrice","TotalLineRevenue","Is_UK","IsGuest"]].sample(6, random_state=1)

,InvoiceNo,InvoiceDate,Month,DayOfWeek,Hour,Quantity,UnitPrice,TotalLineRevenue,Is_UK,IsGuest
475795,577258,2011-11-18 12:27:00,11,Friday,12,3,1.65,4.95,1,False
262622,560118,2011-07-15 10:21:00,7,Friday,10,12,1.65,19.80,1,False
100920,544933,2011-02-25 08:44:00,2,Friday,8,6,2.95,17.70,0,False
245395,558770,2011-07-03 15:19:00,7,Sunday,15,1,2.10,2.10,1,False
288127,562377,2011-08-04 15:04:00,8,Thursday,15,2,0.85,1.70,1,False
21518,538146,2010-12-09 16:10:00,12,Thursday,16,8,0.85,6.80,1,False


In [9]:
orders = cl.order_table(df)      # ONE row per invoice - a line is not an order
print(f"{len(df):,} transaction lines  ->  {len(orders):,} genuine product orders")
orders.head()

536,641 transaction lines  ->  19,773 genuine product orders


,InvoiceNo,InvoiceDate,CustomerID,Country,Is_UK,Revenue,Units,Lines,Hour,DayOfWeek,YearMonth,IsGuest
0,536365,2010-12-01 08:26:00,"17,850.00",United Kingdom,1,139.12,40,7,8,Wednesday,2010-12,False
1,536366,2010-12-01 08:28:00,"17,850.00",United Kingdom,1,22.20,12,2,8,Wednesday,2010-12,False
2,536367,2010-12-01 08:34:00,"13,047.00",United Kingdom,1,278.73,83,12,8,Wednesday,2010-12,False
3,536368,2010-12-01 08:34:00,"13,047.00",United Kingdom,1,70.05,15,4,8,Wednesday,2010-12,False
4,536369,2010-12-01 08:35:00,"13,047.00",United Kingdom,1,17.85,3,1,8,Wednesday,2010-12,False


### Output of this notebook
`data/processed/lines_clean.parquet`, `orders.parquet`, `cleaning_log.csv` are produced by `python -m src.pipeline` (same code, reproducible end-to-end).

In [10]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "src.pipeline"], cwd="..", check=True)
print("processed tables refreshed")

pipeline done


processed tables refreshed
